# Model B, multiple runs: DeBERTa with 3 seeds and 2 learning rates

One run is not enough to say DeBERTa beats Model A: a different random seed can move validation accuracy by a few points. This notebook trains DeBERTa **6 times** (learning rate 2e-5 and 1e-5, each with seeds 42, 43, 44; 3 epochs each), saves every run's validation predictions, and reports the mean and spread.

**Before you start:** Runtime, then Change runtime type, then **T4 GPU**. Run the cells in order with Shift+Enter, and run the upload cell only once.

Takes about 35 to 40 minutes. Keep the laptop open and awake the whole time, or the session resets and the results are lost.

## 1. Check the GPU

In [1]:
!nvidia-smi

Thu Nov 19 15:54:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install packages

In [2]:
!pip install -q sentencepiece protobuf

## 3. Upload the data (run once)
Pick `data/processed/pairs_significant.csv`. If a copy already exists, this cell keeps a single file with the right name.

In [3]:
import os
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    if name != 'pairs_significant.csv' and name.startswith('pairs_significant'):
        os.replace(name, 'pairs_significant.csv')
assert os.path.exists('pairs_significant.csv'), 'Upload pairs_significant.csv'
print('Ready:', os.path.getsize('pairs_significant.csv'), 'bytes')

Saving pairs_significant.csv to pairs_significant.csv
Ready: 2644198 bytes


## 4. Settings
The 6 runs are listed in `RUNS`. 3 epochs each, since the first run overfit after epoch 2.

In [4]:
import random, time, zipfile
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

BASE = {
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 64,
    "batch_size": 16,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
}
# 2 learning rates x 3 seeds = 6 runs, about 6 minutes each on a T4.
RUNS = [{"learning_rate": lr, "epochs": 3, "seed": seed}
        for lr in (2e-5, 1e-5) for seed in (42, 43, 44)]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed: int) -> None:
    """Fix every source of randomness so a rerun gives the same result."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print("Device:", DEVICE, "| runs planned:", len(RUNS))

Device: cuda | runs planned: 6


## 5. Load the pairs and the tokenizer
Should print `Train pairs: 3212, validation pairs: 648`.

In [5]:
pairs = pd.read_csv("pairs_significant.csv")
train_df = pairs[pairs["split"] == "train"].reset_index(drop=True)
val_df = pairs[pairs["split"] == "val"].reset_index(drop=True)
print(f"Train pairs: {len(train_df)}, validation pairs: {len(val_df)}")


class PairDataset(Dataset):
    """(headline_a, headline_b, label); swap=True also adds every pair reversed."""

    def __init__(self, df: pd.DataFrame, swap: bool):
        a, b, y = list(df["headline_a"]), list(df["headline_b"]), list(df["label"])
        if swap:
            a, b, y = a + b, b + a, y + [1 - v for v in y]
        self.a, self.b, self.y = a, b, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.a[i], self.b[i], self.y[i]


tokenizer = AutoTokenizer.from_pretrained(BASE["model_name"])


def collate(batch):
    """Tokenise a batch as '[CLS] A [SEP] B [SEP]'."""
    a, b, y = zip(*batch)
    enc = tokenizer(list(a), list(b), truncation=True, max_length=BASE["max_length"],
                    padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(y)
    return enc

Train pairs: 3212, validation pairs: 648


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

## 6. Training functions
`train_one` loads a fresh DeBERTa **in float32** for every run, trains it, and keeps the predictions from its best epoch. It stops with an error if the loss ever becomes NaN.

In [6]:
@torch.no_grad()
def predict_proba(model, df: pd.DataFrame) -> np.ndarray:
    """P(headline_a wins), averaged over both orders for symmetry."""
    model.eval()

    def run(a_col, b_col):
        ds = PairDataset(df.rename(columns={a_col: "headline_a", b_col: "headline_b"}), swap=False)
        probs = []
        for batch in DataLoader(ds, batch_size=64, collate_fn=collate):
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            probs.append(torch.softmax(model(**batch).logits.float(), dim=-1)[:, 1].cpu())
        return torch.cat(probs).numpy()

    return (run("headline_a", "headline_b") + (1 - run("headline_b", "headline_a"))) / 2


def train_one(learning_rate: float, epochs: int, seed: int):
    """Train one DeBERTa from scratch; return (history, best val probabilities)."""
    set_seed(seed)
    # float32 on purpose: transformers 5 loads DeBERTa in float16, which gives NaN loss.
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE["model_name"], num_labels=2, dtype=torch.float32).to(DEVICE)
    loader = DataLoader(PairDataset(train_df, swap=True), batch_size=BASE["batch_size"],
                        shuffle=True, collate_fn=collate,
                        generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                                  weight_decay=BASE["weight_decay"])
    total = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(BASE["warmup_ratio"] * total), total)

    history, best_acc, best_probs = [], -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(**batch).loss
            if torch.isnan(loss):
                raise RuntimeError("NaN loss: check the model loaded in float32")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            loss_sum += loss.item()
        probs = predict_proba(model, val_df)
        acc = ((probs > 0.5).astype(int) == val_df["label"].to_numpy()).mean()
        history.append({"learning_rate": learning_rate, "seed": seed, "epoch": epoch,
                        "train_loss": loss_sum / len(loader), "val_accuracy": acc})
        print(f"  epoch {epoch}: train loss {loss_sum / len(loader):.4f}, val accuracy {acc:.3f}")
        if acc > best_acc:
            best_acc, best_probs = acc, probs
    del model, optimizer
    torch.cuda.empty_cache()
    return history, best_probs

## 7. Run all 6 trainings
Prints each epoch. Each run saves its own predictions file.

In [7]:
all_history, summary = [], []
start = time.time()
for i, run in enumerate(RUNS, 1):
    print(f"Run {i}/{len(RUNS)}: lr={run['learning_rate']}, seed={run['seed']}")
    history, probs = train_one(**run)
    all_history += history
    best = max(history, key=lambda h: h["val_accuracy"])
    summary.append({**run, "best_epoch": best["epoch"], "best_val_accuracy": best["val_accuracy"]})
    out = val_df[["test_id", "story_group", "headline_a", "headline_b", "ctr_a", "ctr_b", "label"]].copy()
    out["prob_a_wins"] = probs
    out["prediction"] = (probs > 0.5).astype(int)
    out["correct"] = out["prediction"] == out["label"]
    out.to_csv(f"deberta_val_predictions_lr{run['learning_rate']}_seed{run['seed']}.csv", index=False)
    print(f"  best: epoch {best['epoch']}, val accuracy {best['val_accuracy']:.3f}  "
          f"({(time.time() - start) / 60:.1f} min so far)\n")

Run 1/6: lr=2e-05, seed=42


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

... (14 outputs left out)

  epoch 1: train loss 0.6954, val accuracy 0.591
  epoch 2: train loss 0.5843, val accuracy 0.736
  epoch 3: train loss 0.2626, val accuracy 0.755
  best: epoch 3, val accuracy 0.755  (27.5 min so far)

Run 6/6: lr=1e-05, seed=44


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

  epoch 1: train loss 0.6951, val accuracy 0.610
  epoch 2: train loss 0.6095, val accuracy 0.730
  epoch 3: train loss 0.3110, val accuracy 0.741
  best: epoch 3, val accuracy 0.741  (33.0 min so far)



## 8. Summary and download
Mean and standard deviation of the best validation accuracy for each learning rate. Then everything is zipped and downloaded.

In [8]:
summary_df = pd.DataFrame(summary)
pd.DataFrame(all_history).to_csv("deberta_history_all.csv", index=False)
summary_df.to_csv("deberta_summary.csv", index=False)
table = summary_df.groupby("learning_rate")["best_val_accuracy"].agg(["mean", "std", "min", "max"])
print(table.round(3))

import glob
with zipfile.ZipFile('deberta_runs.zip', 'w') as z:
    for f in glob.glob('deberta_*.csv'):
        z.write(f)
files.download('deberta_runs.zip')

                mean    std    min    max
learning_rate                            
0.00001        0.751  0.008  0.741  0.756
0.00002        0.747  0.007  0.742  0.755


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>